In [1]:
# ================================================================
# PHASE 1 — Baselines and LoRA Fine-Tuning on CIFAR-100
# Research: Curvature-Guided Rank Selection (CGRS) for ViTs
#
# Experiments:
#   A — Full fine-tuning        (upper bound baseline)
#   B — Frozen backbone         (lower bound baseline)
#   C — LoRA at 14 ranks        (main experimental variable)
#
# WHAT IS SAVED (everything Phase 2 and Phase 3 will need):
#   phase1_results/
#     config.json                  — all hyperparameters
#     results.json                 — test acc/loss + param counts
#     lora_r{r}_full.pt            — full model state dict per rank
#     lora_r{r}_adapter/           — lightweight adapter weights
#
# DISK SPACE NOTE:
#   Each lora_r{r}_full.pt is ~350MB.
#   14 ranks = ~4.9GB total. Ensure your CARC scratch allows this.
#   Adapter folders are ~5MB each — negligible.
# ================================================================

In [ ]:
# ----------------------------------------------------------------
# CELL 1 — Imports and reproducibility
# ----------------------------------------------------------------
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from transformers import ViTForImageClassification, ViTImageProcessor
from peft import LoraConfig, get_peft_model

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ----------------------------------------------------------------
# CELL 2 — Config
# All hyperparameters in one place.
# ----------------------------------------------------------------
LORA_RANKS = [3, 5, 6, 10, 12, 16, 30, 52, 64, 80, 100, 128, 150, 200]

CONFIG = {
    # Model
    "model_name"     : "google/vit-base-patch16-224",
    "num_classes"    : 100,

    # Data
    "train_size"     : 45000,

    # Training
    "batch_size"     : 16,
    "epochs"         : 3,
    "weight_decay"   : 0.01,

    # Learning rates
    "lr_full"        : 5e-5,
    "lr_frozen"      : 1e-3,
    "lr_lora"        : 5e-4,

    # LoRA
    "lora_alpha"     : 16,
    "lora_dropout"   : 0.1,
    "target_modules" : ["query", "value"],
    "lora_ranks"     : LORA_RANKS,
}

SAVE_DIR = "./phase1_results"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save config immediately — Phase 2 reads this
with open(f"{SAVE_DIR}/config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print("Config saved.")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# ----------------------------------------------------------------
# CELL 3 — Dataset and DataLoaders
# CIFAR-100: 100 classes, 60000 images
# Split: 45000 train / 5000 val / 10000 test
# ----------------------------------------------------------------
processor = ViTImageProcessor.from_pretrained(CONFIG["model_name"])

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=processor.image_mean,
        std=processor.image_std
    ),
])

print("Loading CIFAR-100...")
full_train   = datasets.CIFAR100(root="./data", train=True,  download=True, transform=transform)
test_dataset = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform)

val_size = len(full_train) - CONFIG["train_size"]
train_dataset, val_dataset = random_split(
    full_train,
    [CONFIG["train_size"], val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

print(f"  Train : {len(train_dataset):,}")
print(f"  Val   : {len(val_dataset):,}")
print(f"  Test  : {len(test_dataset):,}")

In [ ]:
# ----------------------------------------------------------------
# CELL 4 — Model builders and training helpers
# ----------------------------------------------------------------
def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total


def build_full_model():
    model = ViTForImageClassification.from_pretrained(
        CONFIG["model_name"],
        num_labels=CONFIG["num_classes"],
        ignore_mismatched_sizes=True,
    )
    t, total = count_parameters(model)
    print(f"  Full model | Trainable: {t:,} / {total:,} ({100*t/total:.2f}%)")
    return model.to(device)


def build_frozen_model():
    model = ViTForImageClassification.from_pretrained(
        CONFIG["model_name"],
        num_labels=CONFIG["num_classes"],
        ignore_mismatched_sizes=True,
    )
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True
    t, total = count_parameters(model)
    print(f"  Frozen model | Trainable: {t:,} / {total:,} ({100*t/total:.4f}%)")
    return model.to(device)


def build_lora_model(r):
    base_model = ViTForImageClassification.from_pretrained(
        CONFIG["model_name"],
        num_labels=CONFIG["num_classes"],
        ignore_mismatched_sizes=True,
    )
    lora_config = LoraConfig(
        r              = r,
        lora_alpha     = CONFIG["lora_alpha"],
        lora_dropout   = CONFIG["lora_dropout"],
        target_modules = CONFIG["target_modules"],
        bias           = "none",
    )
    model = get_peft_model(base_model, lora_config)
    t, total = count_parameters(model)
    print(f"  LoRA r={r:<4} | Trainable: {t:>10,} / {total:,} ({100*t/total:.3f}%)")
    return model.to(device), t


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    loss_sum, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        outputs   = model(images)
        loss_sum += criterion(outputs.logits, labels).item()
        correct  += (outputs.logits.argmax(dim=1) == labels).sum().item()
        total    += labels.size(0)
    return loss_sum / len(loader), 100.0 * correct / total


def train_model(model, lr, label=""):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=CONFIG["weight_decay"],
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CONFIG["epochs"] * len(train_loader)
    )
    for epoch in range(1, CONFIG["epochs"] + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for step, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                max_norm=1.0
            )
            optimizer.step()
            scheduler.step()
            running_loss += loss.item()
            correct      += (outputs.logits.argmax(dim=1) == labels).sum().item()
            total        += labels.size(0)
            if (step + 1) % 500 == 0:
                print(f"  [{label}] Epoch {epoch} | Step {step+1}/{len(train_loader)} | Loss: {running_loss/(step+1):.4f}")

        val_loss, val_acc = evaluate(model, val_loader)
        print(
            f"  [{label}] Epoch {epoch}/{CONFIG['epochs']} | "
            f"Train Loss: {running_loss/len(train_loader):.4f} "
            f"Train Acc: {100*correct/total:.2f}% | "
            f"Val Loss: {val_loss:.4f} Val Acc: {val_acc:.2f}%"
        )
    return model

print("All helpers ready.")

In [ ]:
# ================================================================
# EXPERIMENT A — Full Fine-Tuning
# Upper bound: every parameter in ViT is updated.
# ================================================================

# ----------------------------------------------------------------
# CELL 5 — Run Experiment A
# ----------------------------------------------------------------
print("\n" + "="*60)
print("  EXPERIMENT A — Full Fine-Tuning")
print("="*60)

model_A = build_full_model()
model_A = train_model(model_A, lr=CONFIG["lr_full"], label="Full")
loss_A, acc_A = evaluate(model_A, test_loader)
print(f"\n  Result -> Test Loss: {loss_A:.4f} | Test Acc: {acc_A:.2f}%")

del model_A
torch.cuda.empty_cache()

In [ ]:
# ----------------------------------------------------------------
# CELL 6 — Save Experiment A
# ----------------------------------------------------------------
results = {
    "Full fine-tune": {
        "test_loss"        : loss_A,
        "test_acc"         : acc_A,
        "trainable_params" : 86567396,   # all params
    }
}
with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Experiment A saved -> results.json")

In [ ]:
# ----------------------------------------------------------------
# CELL 7 — Resume point: load Experiment A (new session)
# ----------------------------------------------------------------
with open(f"{SAVE_DIR}/results.json") as f:
    results = json.load(f)
with open(f"{SAVE_DIR}/config.json") as f:
    CONFIG = json.load(f)

LORA_RANKS = CONFIG["lora_ranks"]
print("Loaded results:")
for k, v in results.items():
    print(f"  {k:<22} -> Test Acc: {v['test_acc']:.2f}%")

In [ ]:
# ================================================================
# EXPERIMENT B — Frozen Backbone
# Lower bound: only classification head is trained.
# ================================================================

# ----------------------------------------------------------------
# CELL 8 — Run Experiment B
# ----------------------------------------------------------------
print("\n" + "="*60)
print("  EXPERIMENT B — Frozen Backbone")
print("="*60)

model_B = build_frozen_model()
trainable_B = sum(p.numel() for p in model_B.parameters() if p.requires_grad)
model_B = train_model(model_B, lr=CONFIG["lr_frozen"], label="Frozen")
loss_B, acc_B = evaluate(model_B, test_loader)
print(f"\n  Result -> Test Loss: {loss_B:.4f} | Test Acc: {acc_B:.2f}%")

del model_B
torch.cuda.empty_cache()

In [ ]:
# ----------------------------------------------------------------
# CELL 9 — Save Experiment B
# ----------------------------------------------------------------
results["Frozen backbone"] = {
    "test_loss"        : loss_B,
    "test_acc"         : acc_B,
    "trainable_params" : trainable_B,
}
with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Experiment B saved -> results.json")

In [ ]:
# ----------------------------------------------------------------
# CELL 10 — Resume point: load Experiments A+B (new session)
# ----------------------------------------------------------------
with open(f"{SAVE_DIR}/results.json") as f:
    results = json.load(f)
with open(f"{SAVE_DIR}/config.json") as f:
    CONFIG = json.load(f)

LORA_RANKS = CONFIG["lora_ranks"]
print("Loaded results:")
for k, v in results.items():
    print(f"  {k:<22} -> Test Acc: {v['test_acc']:.2f}%")

In [ ]:
# ================================================================
# EXPERIMENT C — LoRA at Multiple Ranks
#
# KEY CHANGE vs original Phase 1:
#   We now save the FULL model state dict (.pt) for every rank.
#   This allows Phase 2 to load weights directly and skip
#   re-training entirely — saving 6-8 hours of GPU compute.
#
# Saved per rank:
#   lora_r{r}_full.pt     — full model state dict (~350MB)
#                           needed by Phase 2 for curvature
#   lora_r{r}_adapter/    — PEFT adapter weights only (~5MB)
#                           needed by Phase 3 for CGRS init
#   results.json          — updated with metrics + param count
# ================================================================

# ----------------------------------------------------------------
# CELL 11 — Run Experiment C (all ranks, resume-safe)
# ----------------------------------------------------------------
print("\n" + "="*60)
print("  EXPERIMENT C — LoRA at Multiple Ranks")
print("="*60)

# Estimate disk usage
est_gb = len(LORA_RANKS) * 0.35
print(f"\n  Ranks to train : {LORA_RANKS}")
print(f"  Est. disk usage: ~{est_gb:.1f} GB for .pt checkpoints")
print(f"  Est. time      : ~{len(LORA_RANKS) * 30} min on A100")
print()

for r in LORA_RANKS:
    key = f"LoRA r={r}"

    # Check if both results AND checkpoint already exist — skip if so
    ckpt_path    = f"{SAVE_DIR}/lora_r{r}_full.pt"
    adapter_path = f"{SAVE_DIR}/lora_r{r}_adapter"
    if key in results and os.path.exists(ckpt_path):
        print(f"  r={r:<4} already done (acc={results[key]['test_acc']:.2f}%), skipping.")
        continue

    print(f"\n{'='*55}")
    print(f"  Training LoRA r={r}  ({LORA_RANKS.index(r)+1}/{len(LORA_RANKS)})")
    print(f"{'='*55}")
    torch.cuda.empty_cache()

    model_C, trainable_C = build_lora_model(r)
    model_C = train_model(model_C, lr=CONFIG["lr_lora"], label=f"LoRA r={r}")
    loss_C, acc_C = evaluate(model_C, test_loader)
    print(f"\n  r={r} -> Test Loss: {loss_C:.4f} | Test Acc: {acc_C:.2f}%")

    # ---- Save full state dict (Phase 2 curvature needs this) ----
    torch.save(model_C.state_dict(), ckpt_path)
    print(f"  Full checkpoint : {ckpt_path} ({os.path.getsize(ckpt_path)/1e6:.1f} MB)")

    # ---- Save adapter only (Phase 3 CGRS init) ------------------
    model_C.save_pretrained(adapter_path)
    print(f"  Adapter weights : {adapter_path}/")

    # ---- Save metrics to results.json ---------------------------
    results[key] = {
        "test_loss"        : loss_C,
        "test_acc"         : acc_C,
        "trainable_params" : trainable_C,
        "ckpt_path"        : ckpt_path,
        "adapter_path"     : adapter_path,
    }
    with open(f"{SAVE_DIR}/results.json", "w") as f:
        json.dump(results, f, indent=2)
    print(f"  Results saved   : {SAVE_DIR}/results.json")

    del model_C
    torch.cuda.empty_cache()

print("\nAll LoRA ranks complete.")

In [ ]:
# ----------------------------------------------------------------
# CELL 12 — Final save + file verification
# Run after Cell 11 completes. Safe to re-run.
# ----------------------------------------------------------------
with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)

print("=" * 60)
print("  PHASE 1 FILE VERIFICATION")
print("=" * 60)

# Check config
cfg_ok = os.path.exists(f"{SAVE_DIR}/config.json")
print(f"  config.json       : {'ok' if cfg_ok else 'MISSING'}")

# Check results
res_ok = os.path.exists(f"{SAVE_DIR}/results.json")
print(f"  results.json      : {'ok' if res_ok else 'MISSING'}")

# Check each rank checkpoint
total_gb = 0
all_ckpts_ok = True
for r in LORA_RANKS:
    ckpt    = f"{SAVE_DIR}/lora_r{r}_full.pt"
    adapter = f"{SAVE_DIR}/lora_r{r}_adapter"
    ckpt_ok    = os.path.exists(ckpt)
    adapter_ok = os.path.isdir(adapter)
    if ckpt_ok:
        mb = os.path.getsize(ckpt) / 1e6
        total_gb += mb / 1000
        print(f"  lora_r{r}_full.pt : ok ({mb:.0f} MB) | adapter: {'ok' if adapter_ok else 'MISSING'}")
    else:
        print(f"  lora_r{r}_full.pt : MISSING  | adapter: {'ok' if adapter_ok else 'MISSING'}")
        all_ckpts_ok = False

print(f"\n  Total checkpoint size : {total_gb:.2f} GB")
print()
if all_ckpts_ok:
    print("  All files present. Phase 2 can proceed without retraining.")
else:
    print("  Some checkpoints missing — re-run Cell 11.")

print("\n  Zip command:")
print("    zip -r phase1_final.zip phase1_results/")

In [ ]:
# ----------------------------------------------------------------
# CELL 13 — Resume point: load all results (new session)
# ----------------------------------------------------------------
with open(f"{SAVE_DIR}/results.json") as f:
    results = json.load(f)
with open(f"{SAVE_DIR}/config.json") as f:
    CONFIG = json.load(f)

LORA_RANKS = CONFIG["lora_ranks"]
print("All Phase 1 results loaded:")
for k, v in results.items():
    params = v.get('trainable_params', 'n/a')
    print(f"  {k:<22} -> Test Acc: {v['test_acc']:.2f}% | Test Loss: {v['test_loss']:.4f} | Params: {params:>12,}")

In [ ]:
# ----------------------------------------------------------------
# CELL 14 — Phase 1 Summary Table and Analysis
# ----------------------------------------------------------------
print("=" * 80)
print("  PHASE 1 COMPLETE RESULTS")
print("=" * 80)
print(f"{'Method':<22} {'Trainable Params':>18} {'% of Total':>12} {'Test Loss':>10} {'Test Acc':>10}")
print("-" * 80)

TOTAL_PARAMS = 86567396  # ViT-Base

order = ["Full fine-tune", "Frozen backbone"] + [f"LoRA r={r}" for r in LORA_RANKS]
for method in order:
    if method not in results:
        print(f"  {method:<20} {'—':>18} {'—':>12} {'not run':>10} {'—':>10}")
        continue
    v      = results[method]
    n_par  = v.get("trainable_params", 0)
    pct    = 100 * n_par / TOTAL_PARAMS if n_par else 0
    print(f"  {method:<20} {n_par:>18,} {pct:>11.3f}% {v['test_loss']:>10.4f} {v['test_acc']:>9.2f}%")

print("=" * 80)

# ---- Rank vs accuracy trend -----------------------------------
full_acc   = results["Full fine-tune"]["test_acc"]
frozen_acc = results["Frozen backbone"]["test_acc"]

completed = [
    (r, results[f"LoRA r={r}"]["test_acc"])
    for r in LORA_RANKS if f"LoRA r={r}" in results
]

if completed:
    print("\n  LoRA Rank vs Test Accuracy (rank-to-rank gains):")
    print("  " + "-" * 52)
    prev_acc = None
    for r, acc in sorted(completed, key=lambda x: x[0]):
        gap = full_acc - acc
        if prev_acc is not None:
            gain = acc - prev_acc
            flag = "  <- diminishing" if gain < 0.3 else ""
            print(f"  r={r:<5} {acc:>6.2f}%  (+{gain:.2f}%)  gap={gap:.2f}%{flag}")
        else:
            print(f"  r={r:<5} {acc:>6.2f}%  (baseline)  gap={gap:.2f}%")
        prev_acc = acc

    best_r, best_acc   = max(completed, key=lambda x: x[1])
    worst_r, worst_acc = min(completed, key=lambda x: x[1])

    print(f"\n  Upper bound (Full fine-tune)   : {full_acc:.2f}%")
    print(f"  Lower bound (Frozen backbone)  : {frozen_acc:.2f}%")
    print(f"  Best  LoRA  (r={best_r})        : {best_acc:.2f}%  (gap: {full_acc-best_acc:.2f}%)")
    print(f"  Worst LoRA  (r={worst_r})       : {worst_acc:.2f}%  (gap: {full_acc-worst_acc:.2f}%)")

print("=" * 80)
print("\nPhase 1 complete. All checkpoints saved. Proceed to Phase 2.")
print(f"Phase 2 will load weights from: {SAVE_DIR}/lora_r{{r}}_full.pt")

---
# PHASE 1 — Visualisation Cells
Run these cells after Cell 13 (or Cell 14). They load `results.json` independently and do not require any re-training.

**Plots generated:**
- Plot 1+2 (merged): Test Accuracy & Test Loss vs LoRA Rank (dual-axis)
- Plot 5: Test Loss vs Test Accuracy scatter (all methods)


In [ ]:
pip install matplotlib


In [ ]:
# ================================================================
# PHASE 1 — PLOT SETUP
# Loads results.json and prepares data for all Phase 1 plots.
# Safe to run in a fresh session (no GPU needed).
# ================================================================

# ----------------------------------------------------------------
# PLOT CELL 1 — Imports and data load
# ----------------------------------------------------------------
import os
import json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

matplotlib.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 12,
    'xtick.labelsize'  : 10,
    'ytick.labelsize'  : 10,
    'legend.fontsize'  : 10,
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
})

PHASE1_DIR  = './phase1_results'
PLOTS_DIR   = './plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

with open(f'{PHASE1_DIR}/results.json') as f:
    results = json.load(f)

LORA_RANKS = [3, 5, 6, 10, 12, 16, 30, 52, 64, 80, 100, 128, 150, 200]

# --- extract LoRA series ---
lora_ranks  = [r for r in LORA_RANKS if f'LoRA r={r}' in results]
lora_accs   = [results[f'LoRA r={r}']['test_acc']  for r in lora_ranks]
lora_losses = [results[f'LoRA r={r}']['test_loss'] for r in lora_ranks]
lora_params = [results[f'LoRA r={r}']['trainable_params'] for r in lora_ranks]

full_acc    = results['Full fine-tune']['test_acc']
full_loss   = results['Full fine-tune']['test_loss']
full_params = results['Full fine-tune']['trainable_params']

frozen_acc    = results['Frozen backbone']['test_acc']
frozen_loss   = results['Frozen backbone']['test_loss']
frozen_params = results['Frozen backbone']['trainable_params']

print(f'Loaded {len(lora_ranks)} LoRA ranks: {lora_ranks}')
print(f'Full FT  : acc={full_acc:.2f}%  loss={full_loss:.4f}')
print(f'Frozen   : acc={frozen_acc:.2f}%  loss={frozen_loss:.4f}')
print(f'Plots dir: {PLOTS_DIR}')


In [ ]:
# ----------------------------------------------------------------
# PLOT CELL 2 — Plot 1+2 (Merged): Test Accuracy & Test Loss vs LoRA Rank
# Dual y-axis: accuracy (left, blue) + loss (right, red)
# Reference lines: Full FT ceiling and Frozen backbone floor
# ----------------------------------------------------------------

fig, ax1 = plt.subplots(figsize=(12, 6))

COLOR_ACC  = '#1a6fab'
COLOR_LOSS = '#c0392b'
COLOR_FULL = '#2ecc71'
COLOR_FRZ  = '#e67e22'

x = np.arange(len(lora_ranks))
tick_labels = [str(r) for r in lora_ranks]

# --- Accuracy (left axis) ---
ax1.plot(x, lora_accs, color=COLOR_ACC, marker='o', linewidth=2.2,
         markersize=7, label='Test Accuracy', zorder=3)
ax1.fill_between(x, lora_accs, alpha=0.08, color=COLOR_ACC)
ax1.set_xlabel('LoRA Rank  (r)', labelpad=8)
ax1.set_ylabel('Test Accuracy (%)', color=COLOR_ACC, labelpad=8)
ax1.tick_params(axis='y', labelcolor=COLOR_ACC)
ax1.set_xticks(x)
ax1.set_xticklabels(tick_labels)
ax1.set_ylim(82, 95.5)

# Reference lines on acc axis
ax1.axhline(full_acc,   linestyle='--', color=COLOR_FULL, linewidth=1.6, alpha=0.85)
ax1.axhline(frozen_acc, linestyle=':',  color=COLOR_FRZ,  linewidth=1.6, alpha=0.85)
ax1.text(len(lora_ranks)-0.1, full_acc + 0.12,
         f'Full FT ceiling  {full_acc:.2f}%', color=COLOR_FULL,
         ha='right', va='bottom', fontsize=9.5)
ax1.text(len(lora_ranks)-0.1, frozen_acc - 0.25,
         f'Frozen floor  {frozen_acc:.2f}%', color=COLOR_FRZ,
         ha='right', va='top', fontsize=9.5)

# Annotate saturation zone
sat_idx = lora_ranks.index(52)  # saturation starts ~r=52
ax1.axvspan(sat_idx, len(lora_ranks)-1, alpha=0.05, color='gray', label='Saturation zone')
ax1.text(sat_idx + 0.15, 84.0, 'Saturation zone', color='gray', fontsize=8.5, style='italic')

# --- Loss (right axis) ---
ax2 = ax1.twinx()
ax2.plot(x, lora_losses, color=COLOR_LOSS, marker='s', linewidth=2.2,
         markersize=6, linestyle='--', label='Test Loss', zorder=3)
ax2.set_ylabel('Test Loss', color=COLOR_LOSS, labelpad=8)
ax2.tick_params(axis='y', labelcolor=COLOR_LOSS)
ax2.set_ylim(0.0, 1.3)

# Reference loss lines
ax2.axhline(full_loss,   linestyle='--', color=COLOR_FULL, linewidth=1.2, alpha=0.5)
ax2.axhline(frozen_loss, linestyle=':',  color=COLOR_FRZ,  linewidth=1.2, alpha=0.5)

# --- Legend ---
legend_elements = [
    Line2D([0],[0], color=COLOR_ACC,  marker='o', linewidth=2, markersize=7, label='Test Accuracy (left axis)'),
    Line2D([0],[0], color=COLOR_LOSS, marker='s', linewidth=2, markersize=6, linestyle='--', label='Test Loss (right axis)'),
    Line2D([0],[0], color=COLOR_FULL, linewidth=1.6, linestyle='--', label=f'Full FT: acc={full_acc:.2f}%'),
    Line2D([0],[0], color=COLOR_FRZ,  linewidth=1.6, linestyle=':',  label=f'Frozen: acc={frozen_acc:.2f}%'),
]
ax1.legend(handles=legend_elements, loc='lower right', framealpha=0.9)

ax1.set_title('Test Accuracy & Test Loss vs LoRA Rank  (14-Rank Sweep, CIFAR-100, ViT-Base/16)',
              fontsize=13, fontweight='bold', pad=12)
ax1.grid(True, axis='both', linestyle='--', alpha=0.35, zorder=0)

plt.tight_layout()
out_path = f'{PLOTS_DIR}/plot_1_2_acc_loss_vs_rank.png'
fig.savefig(out_path)
plt.show()
print(f'Saved -> {out_path}')


In [ ]:
# ----------------------------------------------------------------
# PLOT CELL 3 — Plot 5: Test Loss vs Test Accuracy (All Methods)
# Scatter placing every method in loss-accuracy space.
# Baselines (Full FT, Frozen) highlighted separately.
# ----------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 7))

# Color LoRA points by rank value (continuous colormap)
cmap  = plt.cm.viridis
norm  = plt.Normalize(vmin=min(lora_ranks), vmax=max(lora_ranks))
colors = [cmap(norm(r)) for r in lora_ranks]

sc = ax.scatter(lora_losses, lora_accs, c=lora_ranks, cmap=cmap,
                norm=norm, s=110, zorder=4, edgecolors='white', linewidths=0.6)

# Annotate each LoRA point
for i, r in enumerate(lora_ranks):
    offset_x = 0.01
    offset_y = 0.12
    # nudge a few crowded labels
    if r in [5, 6]:
        offset_y = -0.25
    if r in [64, 80]:
        offset_x = -0.03
        offset_y = -0.25
    ax.annotate(f'r={r}',
                (lora_losses[i], lora_accs[i]),
                xytext=(lora_losses[i] + offset_x, lora_accs[i] + offset_y),
                fontsize=8.5, color='#333333',
                arrowprops=dict(arrowstyle='-', color='#aaaaaa', lw=0.7))

# Full FT baseline
ax.scatter([full_loss], [full_acc], color='#2ecc71', s=200, marker='*',
           zorder=5, edgecolors='white', linewidths=0.8, label=f'Full Fine-Tune ({full_acc:.2f}%)')
ax.annotate('Full FT', (full_loss, full_acc),
            xytext=(full_loss + 0.01, full_acc - 0.4),
            fontsize=9, color='#1a7a40', fontweight='bold')

# Frozen baseline
ax.scatter([frozen_loss], [frozen_acc], color='#e67e22', s=160, marker='D',
           zorder=5, edgecolors='white', linewidths=0.8, label=f'Frozen Backbone ({frozen_acc:.2f}%)')
ax.annotate('Frozen', (frozen_loss, frozen_acc),
            xytext=(frozen_loss + 0.01, frozen_acc + 0.25),
            fontsize=9, color='#c0681a', fontweight='bold')

# Trend arrow (direction of improvement)
ax.annotate('', xy=(0.25, 91.5), xytext=(0.9, 85.5),
            arrowprops=dict(arrowstyle='->', color='#888888', lw=1.5, linestyle='dashed'))
# ax.text(0.55, 88.2, 'Increasing rank →', fontsize=9, color='#666666', style='italic', rotation=-22)

cbar = fig.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label('LoRA Rank  (r)', fontsize=10)

ax.set_xlabel('Test Loss', labelpad=8)
ax.set_ylabel('Test Accuracy (%)', labelpad=8)
ax.set_title('Test Loss vs Test Accuracy — All Methods  (CIFAR-100, ViT-Base/16)',
             fontsize=13, fontweight='bold', pad=12)
ax.legend(loc='lower left', framealpha=0.9)
ax.grid(True, linestyle='--', alpha=0.35)

plt.tight_layout()
out_path = f'{PLOTS_DIR}/plot_5_loss_vs_acc_all_methods.png'
fig.savefig(out_path)
plt.show()
print(f'Saved -> {out_path}')
